In [ ]:
import wfdb
import numpy as np
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
DATA_PATH = "mit-bih-arrhythmia-database-1.0.0"

label_map = {
    "N": 0,
    "V": 1,
    "A": 2,
    "R": 3,
    "L": 4,
    "/": 5,
    "~": 6,
    "+": 7
}

# 저장
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)

# =========================
# RECORD 처리
# =========================
def process_record(record_name):
    record = wfdb.rdrecord(os.path.join(DATA_PATH, record_name))
    ann = wfdb.rdann(os.path.join(DATA_PATH, record_name), 'atr')

    signal = record.p_signal  # (T, 2)
    pos = ann.sample
    sym = ann.symbol

    return signal, pos, sym

# =========================
# SEGMENTATION
# =========================
def segment_by_event(signal, pos, sym, record_name):

    td_list = []
    labels = []
    meta = []

    start_idx = 0

    for i in range(1, len(sym)):
        if sym[i] != sym[i-1]:
            # segment boundary
            start_sample = pos[start_idx]
            end_sample = pos[i]

            if end_sample - start_sample > 10:  # 너무 짧은 건 제거
                segment = signal[start_sample:end_sample, :2].T  # (2, W)

                label = label_map.get(sym[i-1], None)
                if label is not None:
                    td_list.append(segment)
                    labels.append(label)

                    meta.append({
                        "record": record_name,
                        "start": int(start_sample),
                        "end": int(end_sample),
                        "symbol": sym[i-1],
                        "label": label
                    })

            start_idx = i

    return td_list, labels, meta

# =========================
# MAIN
# =========================
def build_dataset(records):

    td_all = []
    labels_all = []
    meta_all = []

    for rec in records:
        print(f"Processing {rec}")
        signal, pos, sym = process_record(rec)

        td, lb, meta = segment_by_event(signal, pos, sym, rec)

        td_all.extend(td)
        labels_all.extend(lb)
        meta_all.extend(meta)

    # ⚠️ variable length → padding 필요
    max_len = max([x.shape[1] for x in td_all])

    padded = []
    for x in td_all:
        pad_width = max_len - x.shape[1]
        padded_x = np.pad(x, ((0,0),(0,pad_width)))
        padded.append(padded_x)

    td_all = np.array(padded)
    labels_all = np.array(labels_all)

    return td_all, labels_all, meta_all

# =========================
# RUN
# =========================
records = ["100", "101", "102","103","104","105","106","107","108","109",
"111","112","113","114","115","116","117","118","119",
"121","122","123","124",
"200","201","202","203","205","207","208","209","210",
"212","213","214","215","217",
"219","220","221","222","223",
"228","230","231","232","233","234"]

td_data, labels, meta = build_dataset(records)

np.savez("td_shard_000.npz", data=td_data)
np.savez("label_shard_000.npz", labels=labels)

pd.DataFrame(meta).to_csv("metadata.csv", index=False)

print("TD:", td_data.shape)
print("Labels:", labels.shape)